# 라우터 ELECTRA 분류기 — 파인튜닝 파이프라인 (Colab)

**⚠️ 실행 전 반드시 읽을 것 — 학습셋 교체 + 체크포인트 선택 방식 교정 버전입니다.**

이 노트북은 `train_router_electra (1).ipynb` 베이스라인에서 두 가지를 바꿨습니다. 나머지 하이퍼파라미터(학습률·배치사이즈·에폭·시드 등)는 베이스라인과 동일합니다.

1. **학습셋 교체**: `router_trainset_v2_natural450_merged.jsonl` (in-scope 전용, 2,115건 = `router_trainset_v2.jsonl` 1,665건 + `testset_natural_450.jsonl` 450건 병합). 필드: `question`(텍스트), `business_function`(라벨, 실제 6업무). 두 원본 파일 모두 동일한 `question`/`business_function` 스키마라 병합 시 라벨 매핑 이슈는 없습니다(라벨셋 6종 완전 일치 확인됨). `is_oos`/`group` 없음(전부 in-scope).
2. **체크포인트 선택 방식 교정** (`router_finetune_record.md` §3-2-1~3): 베이스라인은 `load_best_model_at_end` + `metric_for_best_model="macro_f1"`로 저장 시점에 "최고" 체크포인트를 확정했는데, 이는 레코드가 v1의 실수로 지목한 방식입니다 — 실제 성능축(dev의 G5b AUROC, 근접 OOD 분리력)과 val macro_f1은 다른 축이라 학습이 진행될수록 어긋납니다. 이 노트북은 EarlyStopping/`load_best_model_at_end`를 끄고 `EPOCHS`를 끝까지 돌려 에폭별 체크포인트를 전부 저장(`save_only_model=True`, `save_total_limit=None`)한 뒤, **dev(`TEST_PATH`, 1150)의 G5b AUROC로 사후 선택**합니다("학습 1회 = EPOCHS만큼의 후보 지점"). dev(`TEST_PATH`)가 없으면 val macro_f1로 폴백하며, 이 경우 화면에 경고가 출력됩니다.

평가셋 스키마 노트(레코드 §3-1 참고):

- **평가셋 = `testset_router_oos_1150.jsonl`** (MSP/AUROC 및 체크포인트 선택용, 선택 파일이지만 없으면 체크포인트 선택이 val macro_f1로 대체됨). 필드: `text`, `domain`(D1~D6), `is_oos`, `group`(G5a=far/G5b=near). **학습셋과 라벨 체계가 다릅니다**(업무명 vs D코드).
- 두 스키마는 **헬퍼(`get_text`/`get_label`)로 모두 처리**합니다. near-OOD AUROC(G5b)는 라벨 매핑 없이 바로 나오지만(is_oos·group만 사용), 평가셋의 6-class 도메인 정확도는 학습 라벨(업무명)과 평가 라벨(D코드)이 안 맞아 자동 skip됩니다(D코드↔업무 매핑 확정 전까지).
- 4개 후보는 전부 `AutoModelForSequenceClassification` 호환이라 `SELECTED_MODEL_KEY`만 바꾸면 재실행됩니다. `dialog-koelectra`는 small(~14M)만 공개라 나머지 base(110M)와 규모 차이 감안.


In [ ]:
# 1. 환경 설정 — 버전 고정(코랩 기본 transformers와 TrainingArguments 인자명이 달라 충돌하는 걸 방지)
!pip install -q -U transformers==4.44.2 accelerate==0.33.0 scikit-learn

**⚠️ 위 pip install 셀 실행 직후, 바로 아래 셀을 실행해 런타임을 강제 재시작하세요.**

`ModuleNotFoundError: No module named 'numpy.rec'` 같은 오류는 코드 문제가 아니라, 같은 커널 안에서 numpy 계열 패키지를 pip install한 뒤 재시작 없이 바로 import해서 생기는 코랩 특유의 증상입니다(이미 로드된 numpy C-확장과 새로 깔린 numpy 바이너리가 어긋남). 아래 셀이 프로세스를 강제 종료시켜 코랩이 새 런타임으로 자동 재시작하게 만듭니다 — 재시작 후에는 pip install 셀을 다시 실행할 필요 없이 **다음 셀(임포트)부터 이어서 실행**하세요.

In [ ]:
import os
os.kill(os.getpid(), 9)  # 코랩이 자동으로 새 런타임을 띄운다. 이후 다음 셀부터 이어서 실행.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# 스키마 흡수 헬퍼 — 450(question/business_function)과 1150(text/domain) 모두 처리
def get_text(r: dict) -> str:
    return r.get("text") or r.get("question")


def get_label(r: dict):
    # 학습셋은 business_function, 평가셋(1150)은 domain
    return r.get("business_function") or r.get("domain")


def is_oos(r: dict) -> bool:
    return bool(r.get("is_oos", False))  # 450엔 필드 없음 → 전부 in-scope


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# 2. 설정 — 여기만 바꿔서 4개 후보 모델을 스왑

MODEL_CANDIDATES = {
    "koelectra-base": "monologg/koelectra-base-v3-discriminator",
    "koelectra-small": "monologg/koelectra-small-v3-discriminator",
    "kcelectra-base": "beomi/KcELECTRA-base-v2022",
    "kr-electra": "snunlp/KR-ELECTRA-discriminator",
    "dialog-koelectra": "skplanet/dialog-koelectra-small-discriminator",  # small(~14M)만 공개
}

SELECTED_MODEL_KEY = "kr-electra"  # <- 4개 후보 전부 같은 코드로 실험 (여기만 변경)
MODEL_NAME = MODEL_CANDIDATES[SELECTED_MODEL_KEY]
assert MODEL_NAME is not None, f"'{SELECTED_MODEL_KEY}'의 모델 id가 비어 있습니다."

# 데이터 경로 — /content/data 에 아래 파일을 올린 뒤 실행
DATA_DIR = Path("/content/data")
TRAIN_PATH = DATA_DIR / "router_trainset_v2_natural450_merged.jsonl"  # 학습(in-scope, question+business_function, v2+450 병합) — 필수
VAL_PATH = DATA_DIR / "router_val.jsonl"                  # 없으면 TRAIN에서 자동 분할
TEST_PATH = DATA_DIR / "testset_router_oos_1150.jsonl"    # MSP/AUROC 평가(1150, text+is_oos+group) — 선택

MAX_LEN = 64          # 민원체 단문 기준. 질의 길면 96~128
BATCH_SIZE = 16
EPOCHS = 8            # EarlyStopping 미사용 — 8에폭 전부 돌려 체크포인트 8개 확보(7-1 셀에서 사후 선택)
LR = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42             # 다중시드 보고 시 42,1,7,100,2024 등으로 반복
VAL_RATIO = 0.2

OUTPUT_DIR = Path("/content/router_electra_out_v2trainset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print("selected model:", MODEL_NAME)


In [ ]:
# 3. 데이터 로딩 (스키마는 헬퍼로 흡수 — 450/1150 모두 가능)


def load_jsonl(path: Path) -> list:
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


train_rows = load_jsonl(TRAIN_PATH)
assert train_rows, (
    f"{TRAIN_PATH}가 비어있거나 없습니다. router_trainset_v2_natural450_merged.jsonl 을 이 경로에 올린 뒤 다시 실행하세요."
)

val_rows = load_jsonl(VAL_PATH)
if not val_rows:
    random.Random(SEED).shuffle(train_rows)
    n_val = max(1, int(len(train_rows) * VAL_RATIO))
    val_rows, train_rows = train_rows[:n_val], train_rows[n_val:]

test_rows = load_jsonl(TEST_PATH)  # 없으면 8번 셀(MSP/AUROC)은 건너뜀

print(f"train={len(train_rows)}  val={len(val_rows)}  test(mixed)={len(test_rows)}")


In [ ]:
# 4. 라벨 체계 — 학습 데이터의 라벨(business_function/domain)에서 자동 도출

LABELS = sorted({get_label(r) for r in train_rows if get_label(r) and not is_oos(r)})
assert LABELS, "학습 데이터에 in-scope 라벨(business_function 또는 domain)이 없습니다."
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}
print("LABELS:", LABELS)

train_rows = [r for r in train_rows if get_label(r) and not is_oos(r)]
val_rows = [r for r in val_rows if get_label(r) and not is_oos(r)]
print(f"학습에 실제 사용되는 in-scope 건수: train={len(train_rows)}  val={len(val_rows)}")

In [ ]:
# 5. 토크나이즈 + Dataset — 4개 후보 모두 AutoTokenizer로 동일 처리

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class RouterDataset(Dataset):
    def __init__(self, rows: list):
        texts = [get_text(r) for r in rows]
        self.labels = [label2id[get_label(r)] for r in rows]
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "labels": torch.tensor(self.labels[idx]),
        }


train_ds = RouterDataset(train_rows)
val_ds = RouterDataset(val_rows)
print(f"train_ds={len(train_ds)}  val_ds={len(val_ds)}")

In [ ]:
# 6. 모델 로딩

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

# safetensors 저장 시 "non contiguous tensor" 오류 방지(ELECTRA류에서 흔히 나는 증상) — 미리 방어적으로 정리
for _p in model.parameters():
    _p.data = _p.data.contiguous()

In [ ]:
# 7. 학습 — 체크포인트 선택 기준 교정(router_finetune_record.md §3-2-1~3)
#    v1은 val macro_f1로 "최고" 체크포인트를 저장 시점에 확정했는데, 성능축(dev G5b AUROC)과 다른 축이라
#    학습이 진행될수록 과적합이 macro_f1 최고점과 체계적으로 어긋났다(레코드 §3-2-1).
#    v2는 이 결합을 끊는다: EarlyStopping/load_best_model_at_end를 끄고 EPOCHS를 끝까지 돌려
#    에폭별 체크포인트 전부(save_only_model=True, save_total_limit=None)를 남긴 뒤,
#    다음 셀에서 dev(TEST_PATH, 1150)의 G5b AUROC로 사후 선택한다("학습 1회 = EPOCHS만큼의 후보 지점").


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def build_training_args(**kw):
    try:
        return TrainingArguments(eval_strategy="epoch", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kw)


training_args = build_training_args(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    save_total_limit=None,          # 에폭별 체크포인트 전부 보존 — macro_f1로 사전 탈락시키지 않음
    save_only_model=True,           # 재개 학습 안 함 — optimizer state 생략해 용량 절약
    load_best_model_at_end=False,   # macro_f1 기준 자동 선택을 끔(§3-2-1의 v1 실수를 반복하지 않기 위함)
    logging_steps=10,
    seed=SEED,
    report_to="none",
    save_safetensors=False,         # ELECTRA + safetensors 조합에서 나는 "non contiguous tensor" 오류 회피
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    # EarlyStoppingCallback 미사용: load_best_model_at_end=False와 함께 쓸 수 없고,
    # 에폭별 체크포인트 8개를 전부 남기려면 EPOCHS를 끝까지 돌려야 한다(§3-2-3).
)

train_result = trainer.train()
print(trainer.evaluate())


In [ ]:
# 7-1. 체크포인트 사후 선택 — dev(TEST_PATH, 1150)의 G5b AUROC 기준(§3-2-2/3, §5-3)
#      G5b(근접 OOD·핵심 승부처) 표본이 없으면 전체 AUROC로, dev 자체가 없으면 val macro_f1로 폴백한다.


@torch.no_grad()
def predict_msp_with(m, rows: list):
    m.eval()
    texts = [get_text(r) for r in rows]
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    logits = m(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs.argmax(axis=-1), probs.max(axis=-1)


@torch.no_grad()
def macro_f1_on_val(m) -> float:
    m.eval()
    texts = [get_text(r) for r in val_rows]
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    preds = m(**enc).logits.argmax(dim=-1).cpu().numpy()
    labels = np.array([label2id[get_label(r)] for r in val_rows])
    return f1_score(labels, preds, average="macro")


def score_checkpoint(ckpt_dir: Path) -> dict:
    m = AutoModelForSequenceClassification.from_pretrained(ckpt_dir).to(DEVICE)
    result = {"checkpoint": ckpt_dir.name}
    try:
        if test_rows:
            _, msp = predict_msp_with(m, test_rows)
            oos = np.array([is_oos(r) for r in test_rows])
            score = 1 - msp
            neg = ~oos
            pos_g5b = np.array([is_oos(r) and r.get("group") == "G5b" for r in test_rows])
            mask = pos_g5b | neg
            if pos_g5b.sum() >= 1 and neg.sum() >= 1:
                result["metric"] = "G5b_auroc"
                result["value"] = roc_auc_score(oos[mask], score[mask])
            elif oos.any() and (~oos).any():
                result["metric"] = "overall_auroc"
                result["value"] = roc_auc_score(oos, score)
        if "value" not in result:
            result["metric"] = "val_macro_f1_fallback"
            result["value"] = macro_f1_on_val(m)
    finally:
        del m
        torch.cuda.empty_cache()
    return result


ckpt_dirs = sorted(
    (OUTPUT_DIR / "checkpoints").glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)
assert ckpt_dirs, f"{OUTPUT_DIR / 'checkpoints'} 아래에 체크포인트가 없습니다."

if not test_rows:
    print("경고: TEST_PATH(dev, 1150)가 비어 있어 val macro_f1로 폴백합니다 — §3-2-1 문제로 되돌아가는 경로이니 "
          "가능하면 TEST_PATH를 채운 뒤 재실행하세요.")

scores = [score_checkpoint(p) for p in ckpt_dirs]
for s in scores:
    print(f"  {s['checkpoint']:>16s}  {s['metric']:<20s}  {s['value']:.4f}")

best = max(scores, key=lambda s: s["value"])
print(f"\n선택된 체크포인트: {best['checkpoint']}  ({best['metric']}={best['value']:.4f})")

model = AutoModelForSequenceClassification.from_pretrained(
    OUTPUT_DIR / "checkpoints" / best["checkpoint"]
).to(DEVICE)
for _p in model.parameters():
    _p.data = _p.data.contiguous()


In [ ]:
# 8. MSP 기반 OOS 평가 — 선택된 체크포인트(위 7-1 셀) 기준, near-OOD(G5b)가 핵심 지표
#    도메인 정확도는 학습라벨(업무명)과 평가라벨(D코드)이 같을 때만 계산(아니면 skip)


@torch.no_grad()
def predict_msp(rows: list):
    """각 문항에 대해 (argmax index, MSP=max softmax prob) 반환."""
    model.eval()
    texts = [get_text(r) for r in rows]
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs.argmax(axis=-1), probs.max(axis=-1)


def evaluate_router(test_rows: list) -> None:
    if not test_rows:
        print("테스트셋이 비어 있어 MSP/AUROC 평가를 건너뜁니다.")
        return

    pred_idx, msp = predict_msp(test_rows)
    oos = np.array([is_oos(r) for r in test_rows])

    # (a) in-scope 6-class 정확도 — 평가 라벨이 학습 LABELS와 같을 때만
    in_scope_rows = [(i, r) for i, r in enumerate(test_rows)
                     if not is_oos(r) and get_label(r) in label2id]
    if in_scope_rows:
        idxs, rows_ = zip(*in_scope_rows)
        y_true = [label2id[get_label(r)] for r in rows_]
        y_pred = [pred_idx[i] for i in idxs]
        print(f"[in-scope] n={len(y_true)}  accuracy={accuracy_score(y_true, y_pred):.3f}  "
              f"macro_f1={f1_score(y_true, y_pred, average='macro'):.3f}")
        print(confusion_matrix(y_true, y_pred))
    else:
        print("[in-scope] 도메인 정확도 skip — 학습 라벨(업무명)과 평가 라벨(D코드) 체계가 다름. "
              "D코드↔업무 매핑 확정 후 활성화.")

    # (b) OOS 탐지 — MSP 낮을수록 OOS이므로 score=1-msp (라벨 매핑 불필요)
    if oos.any() and (~oos).any():
        score = 1 - msp
        print(f"[OOS 탐지] 전체 AUROC={roc_auc_score(oos, score):.3f}")

        # in-scope(음성) vs 각 OOS 그룹(양성) 분리 — far(G5a)/near(G5b)
        neg = ~oos
        for g in sorted({r.get("group") for r in test_rows if is_oos(r)}):
            pos = np.array([is_oos(r) and r.get("group") == g for r in test_rows])
            m = pos | neg
            if pos.sum() >= 1 and neg.sum() >= 1:
                tag = " <- near-OOD 승부처" if g == "G5b" else (" (far-OOD)" if g == "G5a" else "")
                print(f"   {g} AUROC={roc_auc_score(oos[m], score[m]):.3f}  (n_oos={int(pos.sum())}){tag}")
    else:
        print("테스트셋에 in-scope/OOS가 둘 다 있어야 AUROC를 계산할 수 있습니다.")


evaluate_router(test_rows)

In [ ]:
# 9. 모델 저장 (+ 코랩 밖으로 내보내기용 zip)

final_dir = OUTPUT_DIR / "final"
model.save_pretrained(final_dir, safe_serialization=False)
tokenizer.save_pretrained(final_dir)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR / "router_electra_model_v2trainset"), "zip", final_dir)
print("saved:", zip_path)

# 구글 드라이브에 보관하려면:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/')


## 다음에 할 일

- **4개 후보 비교**: `SELECTED_MODEL_KEY`만 `koelectra-base` → `koelectra-small` → `kcelectra-base` → `kr-electra` → `dialog-koelectra`로 바꿔가며 이 노트북을 그대로 재실행하면 됩니다(§4의 "4모델 비교는 데이터로 확인" 원칙 그대로). `dialog-koelectra`는 small(~14M)만 있어 나머지 base(110M) 후보와 규모 차이가 있다는 점을 감안해서 비교하세요.
- **다중 시드 평균 (routing_model_review.md §7-3 / 8-3, router_finetune_record.md §6-4-1)**: 셀 2의 `SEED`를 바꿔가며(예: 42, 1, 7, 100, 2024) 셀 2~8(+7-1 선택 셀)을 반복 실행하고, 매번 선택된 체크포인트의 `macro_f1`·G5b AUROC를 모아 평균±표준편차로 보고하세요. 단일 시드 최고값만 채택하지 말 것.
- **베이스라인(450만 학습) 대비 비교**: `train_router_electra (1).ipynb`(450만 학습, macro_f1 기준 체크포인트 선택)와 동일 시드·동일 후보로 돌려서 성능 차이가 학습셋 확장 때문인지, 체크포인트 선택 방식 교정 때문인지 분리해서 보세요. 필요하면 학습셋은 그대로 두고 셀 10만 베이스라인 방식으로 되돌려 두 축을 따로 검증할 수 있습니다.
- **실행 전**: `DATA_DIR`(`/content/data`)에 `router_trainset_v2_natural450_merged.jsonl`(필수)·`router_val.jsonl`(선택)·`testset_router_oos_1150.jsonl`(선택, 체크포인트 선택·MSP/AUROC용 — 없으면 val macro_f1로 자동 폴백)을 올려두면 나머지 셀은 수정 없이 그대로 돌아갑니다.
